In [1]:
# EFFECTIVE-BUDGET REPLICATION OF THE BOTTOM-15 MODEL THAT PRODUCED 35% R@1
# Seeds: 13, 21, 67, and 100 (two sequential waves on GPU T4 x2)
#
# Paste this ENTIRE file into ONE Kaggle code cell.
# Select GPU T4 x2.
# Launch with Save Version -> Save & Run All.
#
# The worker below is copied from cell 39 of spladetraining (4).ipynb.
# The original run's selected model was the step-1400 checkpoint; its log
# reached step 1625, so its last validation/model-selection opportunity was
# step 1600. This runner therefore keeps the original 4,506-step LR schedule
# but stops after step 1,600, reproducing the effective selection budget.
# The original seed-42 train/validation split is fixed for every run. Only
# training randomness (shuffle/dropout) changes with seeds 13, 21, 67, 100.
# It launches two seeds at a time so all four use the same Kaggle T4 hardware.

from pathlib import Path
import base64
import hashlib
import os
import re
import subprocess
import sys
import time
import zlib


SEEDS = (42,)
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working")
SCRIPT = OUTPUT_ROOT / "splade_bottom15_old35_exact.py"
ORIGINAL_SCRIPT_SHA256 = "2e8e336be1dee8acfb3f600540d3ec6def275a7512e8881e748039eab55dc48b"
EXPECTED_EFFECTIVE_SCRIPT_SHA256 = "4536153eaf2dbe12116a5ddb4b05e2e7b620de09ed7e8c710d29b1f6386f0bb8"
EFFECTIVE_STOP_STEP = 1600
FIXED_SPLIT_SEED = 42

PAYLOAD = """c-qBUdsExUmj7R$qD!}?R?Zj&hcFNBdTJeTGSmX0fF!$@p;W?ZV=a22mLLqp{p|O5`k`(an~+;mwXPW=b)Q$CKCeElG_T6KVR3y`#kJsGs@_X6WdA41V#xBiSqxbn7fG29S(D~`$o@>Ld79xtEZQc`ym?*btO8(`UNTML7_XioSG0L`&tk!f$_!PYhynapC1$KC>)E3B5*CF-E0#yYd0Whyv@GI`0N#5_2sKSYi7<(q*hsU7XTh7Hd^jlMgx4AYD--jw&Uq~i@T6_Zv+|l3>7TqF%Gbj(;o1ALK8(dRPYw=;?20#0mKHp&BQaa>q|JDAmo|&&F0S*oa>XetsI4QO#Q7IJdz0j04C-c%20ZU?Y`p4+@sqZo4Uno9tSqwowZNX-&YF$8tqmvjRh#poIhHfQK!b#Fl0>nd3;c}@!Fl3itM4m5P7CO)<~MCx^JIKhw|p(yW!W@kKHA=>%d#=}$I5zMM4)V&s%<uswElOnllp$6ZVRBs(k{lnfcAOR0LdSC02NS%Mwb5&x_}y<QTWF?y-LZ%U?ypi7FTRm7W4FKz1mNw(><kh5t1Uk!;gE@gXm~-IK{AE#J9YDEvgI_xG^h1E#U<u+MvJrWyX_Kz<7PnJNfwW%n&lFjYzfL*~$L#!SpOTo}B#-2nqO_`9?&nIIS|?h;SxueeZa(e{#AEEAESC!JBjzRdHI22;SUqs92$RA3vQ%d;2Gbz1o-v8{_T%_pzBWKp+SEpIuiaj79j~yXmO`>7W5~$wlLP2a}bPSxl1>^B?=in53N#=KkyJ`06U-ukXtG8cTa!X35reM0$N7ML$MFHL~nOL?&#u<nn!lNP?$?CI<)KNvGQdr8J=ld54q#i4LYmALKTS-u&X7O%IQ!CzG>JCuA6-aML?IKA7xHql3xeyS)jG@r}2hH$DEi`}-+gx4d_gv)$jL)BS%C#vAX$$z(5@?CyR#B;cRC1JddnzOn6nneKo19m<{VPW~5To8jij`!YE>{B#_F<NilEHrn*O&y$1wz3BbPWEZ1HW>}3*KTl5x`^_dqn|z-3298hmkIwAKHaXELM+94>2%~d$U$*QnE|6H5i06nyVu1jMf8nAbU=d2$yv;H;j}bHR)OpyDWgJdILH(#KcqldZ5vFo_Ha#Xi(5UC_O+WANN|&0oN$e9(6>RA>Ohpvm#%UJAqk;kBncx7dTG+PAWYRYS+OmOHRN>P>MJ$htHqIhv{5d{59>dR>cF?0Z%QOk&USWO>mkJPz5W<9vZbQbuA^=3?wGMX1NnQf$Rdq_SaF5@nGrogJsFXA?6K5(g$n-(eZXoG^jwv8?AU1k{ek>5FnjTsxWk3`ivblel3PCaQ!G(DA4W6a78q3ZD=b4~P>6@7#w>}aZ2#WeiSo1iE5Z!{MMa$LpA$n_GQr)b~+Pn~?8XIQSSlE;qRuv4E^(~<@UIYeX&{wvZma<?@tE-2BMY5mXY>&2wERa?<_orVgK5IBsvEU4Gh+OMG1fYfhV`+it)5(Z(rEJ(kkL1n}YVL7U0uZa8e|`J*o1A_g0t(r=f#YIQVmpi!HO&Rv^sY^Ty1&OZeq#;)+Mu$m1e1kOmyQ&X(5qp>{4SDUW0s_|CO{ljB+q?4n<oB6h_H=HArMUsW76|iyx?^Vfg=Uk2ZM%TaV<C0A0AOVA{<4iG@v#uLW*Ejmv_Q=b($uC1%6PU`wE}<50uNNO~xq}$v6aCN<|R|b%+`<(7+M|8}0i89giffbs&=LWtrSRPn95^C+a{*ng~?@e4eU;d7f?$Mro*1>N%8k0Ty%rL9XfXkK&Kwfh3Vn2NHD$Rw5rB{ZNomQY8k79{ih@*HY%H6ls&*N}Uw^DwfZWAu+l{F&`{@px!FueYCw@5J-3<Z8-Fmw8<O-n%VR7Z!o>M2$Q<3pr`?uRripU)Y9X(zqbJ+gB?8K`FA)8Dm|>SMQMsq*SMmEp^=H~4N{$7$TxAFRuw0!L!y?B=oo2-ylkiRyK!w*E-Z3fX3)V^E}U6+v~1k;D_lsUl^&#J4nW6!rOiMfhbsWAt;qSxoiCow7c}1iz({Tw5;iE&-@-23C^3woxgrUSE$y%ycV~B4<!<f{D;_O-yRYVFHsVKAS)Mf>xDFjA-`8=@1L8Vxt>MJ(IJeo#g!*JY=s}-q7P-!aJp&;mErU(MnhN`kZDK7)C2Iv0A=WVbWk`he?X9h!x24iL;~KMHM{hSxtoP4k{e}Z~*~n<mzv48-5_Nj+nQ>WwB`93pOTew|A1E#CFkdH)+Eu{#K}&&j&@E>CpdT>!cW}~4MiKD0w}vEakBk)k*C7NpG(w$R_TXxDjsj9p7Qy){Z9KV4$vhNs4wVG+I-ZSDXPah5BpT$1v8>6E{IR|$#cAwoK%TBks#~hw(myG-=keEIG-R}0Mx(D-zkVJl1j9GY%GvpjA#t&ehb(;iNCszTEs`xUVka8dLPL?Xg~sPNOR<ZzYs<h&<mI|U6CeY5`^)CmFH-dmxG^VNC44HnEkup5q4un;x!Mgz>p{?$l*bP}CekR3-Q0GPDk#m*)Ddugc_vLLsI`R~fjg)5_i!pX`E-Pe^YnDOx1+6?q}tk+#=LCFv^cF7Ew0~DgGjti=PA}j5txd3rVe|d0@A@dPLU^1??uD&>DROgHlOV*=!VoY<*R8x!7h3YsgO#hs6yRU%fc(L9R_fm;UFuRt~@o9K>w>#M8fARf=Q)K6a|83b5!9J#GDL5qaj6dIbpJ6uZ-%--9A!fk-B*M61>9V*{r)7&shWvM6baS9-3+ELK>@DM-K)01UWPBmYybGpW-UKbIe@~^}5`bLcVw&QAF*LA&vx>aWh+JO2{_IFtjA9T3@LtHA59=x3z&>(T%KJlLUcZAM9Y4#Z38t8bI)&wO%!gv27#_wY12Sg(zz=_Nzu$64eSe)nCu1Siz?F>)Gf{xoJQd{}+v?gJvchxsbtjbyXBtvi~m93{_oC`ym7D*tS&ElvuR$dB&BXazyWMG%v>bQH>XE9#I#FL)15Fr~;Fm=Vg7b@nLurzFsP(4s|aU@_me2<u8$PeVmew@AXkig_~2^&zu&sl9pB!A+Xwvua7r5QH_eFpwm{=mJU1Er4S*JbTnM<$3F>*4u=&~!mBz?QeL<%KWf_XLUnT7T5}PLxIyLXV5=rDauzC}XcmKpKUk_6mx03$6f2M$XM+;b%3WM1S~qSW=?sC+sn<K~7Ev(JT^*T?u7~WF&I1yN4g)z(0*@VT*^-A^VS-m~Fp}rN@+y@*5`c#4rJKXDp(4tUW5wXy?}nlrnn#3g3xVXo|K!2O$f?m`$}JZ%I#4$Zghy4No@<`9f#Niv8mc4d1mVh}Qr)Gf4Ox=r<Ix~gugfu;usqKgN#7)$6zOQt^W@EpNwRb?qwI$62VglUDP@l;tH%c?Lw3j0t3?wfd=}r2-4-hdA?sHng8fFkD&=<3y;(TpX%^U8M+Jfbd(CX`mRgMs)q9Ov^)jVbOm%>%^69rib(v26P}6yI385MeoRMK48#1elst%q?D3`)3=C+J!jry94l=Y3<FBRT8?K-i@;W>6+EJ)WuS3vF(Jg4xbVaq|OO}6DMd{_N`k&G;i%lO&oC(G;C90tc0Jj13u({ChC)-@zHT8Kbb0j4WaTALPa9^62|l`(0xcz*Q+qUb?%&Z>c$cy=9}S9Dnm<O>4}`*Lm<`fVSD35}IPJ7GvU2GuIC<)xePSZc*5vXwkG{vtDt3}E351O(z`Q&T%cQ5tGCTG$Z)gPSntaS;sIkF2tfg?=Pax57=mI6(q}{*SDG74izS+(LjLhX4!{4h<}%i;lV%&-*WZNnA!0<}pHBqt6vQpz1?LGcLyDJ9>YV<I`3LgH9eU`KYGMkQy^xBPPYc#}*_5ioqP#EBlUtP5N$EhI~>6;hU!44N;?2)9-<(QPB3BV67HDRLWgxZ9QP#wL<2Q2PapI$t)2J1{QL*3<lM9KyE5@ROdz?d3;K-Z6o&E*ahn6#LJq;*Pf$n>FGhQ##8(YZ7OOB1G*;Yjn8OMd7OxExs$|$N?sY0XAha$2GQGQnm5;Z^gQTBUv$M{g;#$id#CKMB)hVB4UDSfs0Bf}-JV@Zbbm*!z>b=wV8%Q;v$oj5yv=nkL|1YXge|%)1opck>&1y3AWr;nVAQ>Jl*RdF5(BE|rVd<IV-hLTj<~rKb8hx@plr<xnB}ThLXbm8@od)SfUrr_fsk2~^rX_JD?NSJ0HJYoSEG+=6DV&XlgKLzJh@lta(_bl5Qf^pcm9B`CFj<xX(mkCyb2^F?36l02@GibMjI$jfOK;nh<b*2D$F4quf`1VsC&|)F=GViX|Ka`cVT8((`Z{D2VZLsQs3gBUb?0)8)gffpy}W&NulV%*3Ed}%ll~AwZ|XzEct&c2~t-wE6Zwq{*zbnlqtvBTm7_{;{<n=m6tjr%U*V#B6rwV91f+bb@)e&H6t?9IPWs%Q4H=!kFExgSxtp}>X|iVRKV|z=VF6ud0b6%9?~D6il(f(dc)S`6yH_xg{WS~#TBQ{pT3Akl<VCy%>6%#sRVwIIFGL%6y&2nu<6343e~;0)!_eMf{@|0UyFv9s3_!F#3AbL&`Z*Z-M{r&<q7T466Mi~4CGjj?ZunaWaoZh?w1j;v3?rrz93UyiOn)Ns0=GbS?AHp1HdTU9Q0^dn<I`vf3mBSS53aqT3r;^f%@y#Yj**DV7*T>o~&M7am1D2NrV#um1vGfuni{tgzB=NHlHBYDqFeyUdzU~Lvcvot^7K8%Cw0kGv|b{;ZhoCQnieut7$CtdEMV!CeHDLL-EmR;`Tv}|DiN^Fg1m1<>woY5_c{JJK@{;dTJ<;pHP|c@1v28R!B1XJ{ry@j31nrop1y!=wHt?u^f3yX7CNm>U|fq(z3eRL_Wy?-qqdfwsY=Fx@^V%bp_mmc~Tls#MS7G%Q1gRc<G2swW0a=Z0N4bmWeAUr1Nf;{U<XSZfE5NIQh@utb|*Z%6u=kR;pD*jOvc2Ul<Yi4(+Yd6p}oB+F`zSL}AY_%ko-YljJB}1>N-hbHqNpW96J#;a3pSrH8<Qx#o=D^7@{>7*mJuknQm~uWR)srJ^iv5)1l__$e`Y1&PDOY(ZI=P8_-m0}DGj`oMJ6W$%n2v1l_S+Ol3j&I(RFF>ddEs=B1BdLbER2+(wk_2KvvyJ-=x!Znub?^=2&@jw6?gp@U(LyU#`*x-4WazCtF=}|AVVR^!Nme+4%w>7eY(NZl$iRcNCbA$u=9*WbjR6u`Y@1-%(3cTk7gR$oK`$v-l_E&|2?{@#^ynUT_|6AVt#Cz4kzt4xw!NPJh-_H)ZIlE-@NvxgIUm*^e{>at#*8V8==*vsL>z}>D;vMIq7MkI@4u;M>9h{`D;SS=Ja8^DIH_0-lzo9iSouK~$h6oEV"""

script_bytes = zlib.decompress(base64.b85decode(PAYLOAD.encode("ascii")))
original_script_sha256 = hashlib.sha256(script_bytes).hexdigest()
if original_script_sha256 != ORIGINAL_SCRIPT_SHA256:
    raise RuntimeError(
        "Embedded worker hash mismatch: "
        f"{original_script_sha256} != {ORIGINAL_SCRIPT_SHA256}"
    )

# Preserve the original 4,506-step scheduler/warmup exactly, but stop after
# the last validation opportunity actually reached by the 35% run.
config_marker = b"MAX_OPT_STEPS = None\n"
stop_marker = b"if global_step >= total_steps:\n"
steps_log_marker = b'print("Total optimizer steps:", total_steps)\n'
eta_marker = b"eta = (total_steps - global_step) / max(rate, 1e-9)\n"
split_marker = (
    b"df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)\n"
)
if script_bytes.count(config_marker) != 1:
    raise RuntimeError("Could not locate the unique MAX_OPT_STEPS marker.")
if script_bytes.count(stop_marker) != 1:
    raise RuntimeError("Could not locate the unique training stop marker.")
if script_bytes.count(steps_log_marker) != 1:
    raise RuntimeError("Could not locate the unique total-steps log marker.")
if script_bytes.count(eta_marker) != 1:
    raise RuntimeError("Could not locate the unique ETA marker.")
if script_bytes.count(split_marker) != 1:
    raise RuntimeError("Could not locate the unique train/validation split marker.")
script_bytes = script_bytes.replace(
    config_marker,
    config_marker + f"EFFECTIVE_STOP_STEP = {EFFECTIVE_STOP_STEP}\n".encode("ascii"),
    1,
)
script_bytes = script_bytes.replace(
    stop_marker,
    b"if global_step >= EFFECTIVE_STOP_STEP:\n",
    1,
)
script_bytes = script_bytes.replace(
    steps_log_marker,
    b'print("LR schedule optimizer steps:", total_steps)\n'
    b'print("Execution stop step:", EFFECTIVE_STOP_STEP)\n',
    1,
)
script_bytes = script_bytes.replace(
    eta_marker,
    b"eta = (EFFECTIVE_STOP_STEP - global_step) / max(rate, 1e-9)\n",
    1,
)
script_bytes = script_bytes.replace(
    split_marker,
    (
        "df = df.sample(frac=1.0, "
        f"random_state={FIXED_SPLIT_SEED}).reset_index(drop=True)\n"
        f'print("Fixed train/validation split seed:", {FIXED_SPLIT_SEED})\n'
    ).encode("ascii"),
    1,
)

actual_script_sha256 = hashlib.sha256(script_bytes).hexdigest()
if actual_script_sha256 != EXPECTED_EFFECTIVE_SCRIPT_SHA256:
    raise RuntimeError(
        "Effective-budget worker hash mismatch: "
        f"{actual_script_sha256} != {EXPECTED_EFFECTIVE_SCRIPT_SHA256}"
    )
SCRIPT.write_bytes(script_bytes)
print(
    f"Created old-structure/effective-budget worker: {SCRIPT} "
    f"({SCRIPT.stat().st_size:,} bytes, SHA-256 {actual_script_sha256}); "
    f"original worker SHA-256 {original_script_sha256}",
    flush=True,
)


def canonical_stem(path: Path) -> str:
    return re.sub(r"\s*\(\d+\)$", "", path.stem.lower())


triplet_candidates = [
    path
    for path in INPUT_ROOT.rglob("*.csv")
    if canonical_stem(path) == "training_triplets"
    and (path.parent / "synthetic_pairs_filtered.csv").exists()
]
if not triplet_candidates:
    raise FileNotFoundError(
        "Could not find a directory containing both training_triplets.csv "
        "and synthetic_pairs_filtered.csv. Attach oqtayhasanov/trivlet."
    )

print("Bottom-15 candidates:", flush=True)
for path in triplet_candidates:
    print(f"  {path.parent}", flush=True)

# Use the worker's exact old reconstruction as the artifact fingerprint.
# A candidate is accepted only if it yields:
#   62,515 input triplets
#   53,375 reconstructed triplets
#   9,140 skipped triplets
#   48,038 train / 5,337 validation triplets
bottom15_root = None
preflight_failures = []
for index, triplet_path in enumerate(triplet_candidates):
    candidate = triplet_path.parent
    preflight_output = OUTPUT_ROOT / f"_old35_preflight_{index}"
    command = [
        sys.executable,
        str(SCRIPT),
        "--seed",
        "42",
        "--bottom15-root",
        str(candidate),
        "--output-dir",
        str(preflight_output),
        "--dry-run",
    ]
    completed = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(completed.stdout, flush=True)
    if completed.returncode == 0:
        bottom15_root = candidate
        break
    preflight_failures.append((str(candidate), completed.returncode))

if bottom15_root is None:
    raise RuntimeError(
        "No attached candidate reproduced the exact old-35 data counts. "
        f"Failures: {preflight_failures}"
    )

print(f"Using exact bottom-15 artifact: {bottom15_root}", flush=True)

gpu_lines = [
    line
    for line in subprocess.check_output(["nvidia-smi", "-L"], text=True).splitlines()
    if line.strip()
]
print("\n".join(gpu_lines), flush=True)
if len(gpu_lines) < 2:
    raise RuntimeError(
        f"Select GPU T4 x2. Kaggle exposed only {len(gpu_lines)} GPU(s)."
    )

for seed in SEEDS:
    output_dir = OUTPUT_ROOT / f"old35_exact_seed{seed}"
    if output_dir.exists() and any(output_dir.iterdir()):
        raise RuntimeError(
            f"{output_dir} already contains files. Use a fresh Kaggle version "
            "rather than mixing an old partial run with this exact replication."
        )

def run_wave(wave_number, wave_seeds):
    print(
        f"\nSTARTING WAVE {wave_number}: seeds {tuple(wave_seeds)} "
        f"(effective stop step {EFFECTIVE_STOP_STEP})",
        flush=True,
    )
    jobs = []
    for gpu_index, seed in enumerate(wave_seeds):
        output_dir = OUTPUT_ROOT / f"old35_exact_seed{seed}"
        log_path = OUTPUT_ROOT / f"old35_seed_{seed}.log"
        log_handle = log_path.open("w", buffering=1)
        environment = os.environ.copy()
        environment["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
        environment["CUDA_VISIBLE_DEVICES"] = str(gpu_index)
        environment["PYTHONUNBUFFERED"] = "1"
        environment["TOKENIZERS_PARALLELISM"] = "false"

        command = [
            sys.executable,
            str(SCRIPT),
            "--seed",
            str(seed),
            "--bottom15-root",
            str(bottom15_root),
            "--output-dir",
            str(output_dir),
        ]
        process = subprocess.Popen(
            command,
            env=environment,
            stdout=log_handle,
            stderr=subprocess.STDOUT,
        )
        jobs.append((seed, gpu_index, process, log_handle, log_path, output_dir))
        print(
            f"STARTED old35-effective seed={seed} GPU={gpu_index} "
            f"PID={process.pid} log={log_path}",
            flush=True,
        )

    while any(process.poll() is None for _, _, process, _, _, _ in jobs):
        time.sleep(60)
        print(time.strftime("%Y-%m-%d %H:%M:%S"), flush=True)
        for seed, gpu_index, process, _, log_path, _ in jobs:
            try:
                lines = log_path.read_text(errors="replace").splitlines()
                latest = lines[-1] if lines else "starting..."
            except Exception as error:
                latest = f"log unavailable: {error}"
            state = (
                "RUNNING"
                if process.poll() is None
                else f"EXIT={process.returncode}"
            )
            print(f"seed {seed} {state} | {latest}", flush=True)

    failures = []
    for seed, gpu_index, process, log_handle, log_path, output_dir in jobs:
        return_code = process.wait()
        log_handle.close()
        best_dir = output_dir / "checkpoints" / "best"
        required_best_files = [
            best_dir / "config.json",
            best_dir / "model.safetensors",
            best_dir / "tokenizer_config.json",
            best_dir / "training_state.json",
        ]
        missing_best_files = [
            str(path) for path in required_best_files if not path.exists()
        ]
        print(
            f"FINISHED seed={seed} exit={return_code} "
            f"best_checkpoint={best_dir}",
            flush=True,
        )
        if return_code != 0 or missing_best_files:
            failures.append(
                {
                    "seed": seed,
                    "return_code": return_code,
                    "log": str(log_path),
                    "missing_best_files": missing_best_files,
                }
            )

    if failures:
        raise RuntimeError(f"Wave {wave_number} failed: {failures}")

    print(f"WAVE {wave_number} COMPLETED: seeds {tuple(wave_seeds)}", flush=True)


for wave_index, wave_start in enumerate(range(0, len(SEEDS), 2), start=1):
    run_wave(wave_index, SEEDS[wave_start : wave_start + 2])

completion_file = OUTPUT_ROOT / "OLD35_EFFECTIVE_BUDGET_ALL4_COMPLETED.txt"
completion_file.write_text(
    "\n".join(
        [
            "Old-35 effective-budget replication completed successfully.",
            f"Seeds: {SEEDS}",
            f"Bottom-15 root: {bottom15_root}",
            f"Original worker SHA-256: {original_script_sha256}",
            f"Effective-budget worker SHA-256: {actual_script_sha256}",
            f"Effective stop step: {EFFECTIVE_STOP_STEP}.",
            f"Fixed train/validation split seed: {FIXED_SPLIT_SEED}.",
            "Original LR schedule horizon preserved: 4,506 optimizer steps.",
            "Expected reconstruction: 53,375 used / 9,140 skipped.",
            "Expected split per seed: 48,038 train / 5,337 validation.",
        ]
    )
    + "\n",
    encoding="utf-8",
)

print(
    "\nSUCCESS: old-35 effective-budget replication completed for "
    "seeds 13, 21, 67, and 100.",
    flush=True,
)
print("Download these best-model archives:", flush=True)
for seed in SEEDS:
    print(
        f"  /kaggle/working/old35_seed{seed}_best_checkpoint.zip",
        flush=True,
    )
print(f"Completion manifest: {completion_file}", flush=True)

Created old-structure/effective-budget worker: /kaggle/working/splade_bottom15_old35_exact.py (12,615 bytes, SHA-256 4536153eaf2dbe12116a5ddb4b05e2e7b620de09ed7e8c710d29b1f6386f0bb8); original worker SHA-256 2e8e336be1dee8acfb3f600540d3ec6def275a7512e8881e748039eab55dc48b
Bottom-15 candidates:
  /kaggle/input/datasets/oqtayhasanov/trivlet/kaggle/working/filter_ablation/bottom15_llm
Device: cuda
Triplet path: /kaggle/input/datasets/oqtayhasanov/trivlet/kaggle/working/filter_ablation/bottom15_llm/training_triplets.csv
Pairs path: /kaggle/input/datasets/oqtayhasanov/trivlet/kaggle/working/filter_ablation/bottom15_llm/synthetic_pairs_filtered.csv
Triplet columns: ['query_review_id', 'positive_issue_id', 'negative_issue_id', 'negative_type']
Triplets: 62515
Pairs: 7944
ID-style triplets detected; reconstructing text triplets...
Reconstructed triplets: 53375
Skipped missing text triplets: 9140
Fixed train/validation split seed: 42
Train triplets: 48038
Validation triplets: 5337
DRY RUN PASSE